# 2. When things go wrong

Most of the work in a tool interface for an agent is in the failure cases. Each of these is also covered by a test in `poster-mcp/tests`.

```mermaid
flowchart LR
    A["bad input"] --> A2["error with a next step"]
    B["retry after a timeout"] --> B2["same job, no duplicate"]
    C["too many jobs"] --> C2["refused, with advice"]
    D["job fails"] --> D2["retryable, retry works"]
```

In [1]:
import json
import uuid
from demoUtils import connectMcp, callTool, show, waitForJob

runId = uuid.uuid4().hex[:8]  # makes every run of this notebook independent
baseArgs = dict(
    product_image="https://cdn.example.com/latte.png",
    background_image="https://cdn.example.com/cafe.png",
    prompt=f"Failure demo {runId}",
)

## bad input

Every error says what's wrong and what to do next, so the model can fix itself. The last two are also abuse guards: private network addresses (a classic way to attack internal systems through an image url) and plain `http` are refused.

In [2]:
badInputs = {
    "prompt over 500 characters": {"prompt": "x" * 501},
    "image_count = 9": {"image_count": 9},
    "resolution = 4k": {"resolution": "4k"},
    "private network address": {"product_image": "https://169.254.169.254/latest/meta-data"},
    "plain http": {"background_image": "http://cdn.example.com/bg.png"},
}

async with connectMcp() as session:
    for label, override in badInputs.items():
        result = await callTool(session, "create_poster_job", **{**baseArgs, **override})
        print(f"--- {label}")
        print(result["error"], "\n")

--- prompt over 500 characters
Error executing tool create_poster_job: `prompt` is 501 characters; the maximum is 500. Next step: Shorten the prompt to 500 characters or fewer, keeping the key details, then call create_poster_job again. 

--- image_count = 9
Error executing tool create_poster_job: `image_count` is 9; it must be from 1 to 4. Next step: Use an integer from 1 to 4. If the user wants more, create several jobs of up to 4 each (confirm with the user first). 

--- resolution = 4k
Error executing tool create_poster_job: `resolution` is "4k"; it must be "1k" or "2k". Next step: Use "1k" (fast, default) or "2k" (print quality). 

--- private network address
Error executing tool create_poster_job: `product_image` ('https://169.254.169.254/latest/meta-data') is not an allowed image reference. Next step: Pass an https:// image URL on a public host, or an asset:// reference from an upload. Ask the user for the image if you do not have one. 

--- plain http
Error executing tool creat

## retries

If a request times out, the agent can't know whether the job was created, so it retries. Without protection that makes a second job and doubles the cost. Every create carries an idempotency key:

- the same inputs return the existing job (`duplicate: true`)
- a new key deliberately starts a new job with the same inputs
- reusing a key with different inputs is a conflict

In [3]:
async with connectMcp() as session:
    first = await callTool(session, "create_poster_job", **baseArgs)
    retry = await callTool(session, "create_poster_job", **baseArgs)
    newBatch = await callTool(session, "create_poster_job", **baseArgs, idempotency_key=f"new-batch-{runId}")
    conflict = await callTool(session, "create_poster_job", **{**baseArgs, "prompt": "something else"}, idempotency_key=f"new-batch-{runId}")

print("first call :", first["data"]["job_id"], "duplicate =", first["data"]["duplicate"])
print("retry      :", retry["data"]["job_id"], "duplicate =", retry["data"]["duplicate"], "  <- same job, no extra cost")
print("new key    :", newBatch["data"]["job_id"], "duplicate =", newBatch["data"]["duplicate"], "  <- a deliberate second job")
print("\nsame key, different inputs:")
print(conflict["error"])
assert first["data"]["job_id"] == retry["data"]["job_id"] != newBatch["data"]["job_id"]

first call : eb2f03012e3a duplicate = False
retry      : eb2f03012e3a duplicate = True   <- same job, no extra cost
new key    : ed81ed592a04 duplicate = False   <- a deliberate second job

same key, different inputs:
Error executing tool create_poster_job: This Idempotency-Key was already used with different inputs. Next step: Use a different idempotency_key to start a new job, or repeat the original inputs to get the existing job.


## the job cap

The backend refuses new jobs while too many are processing (5 by default). The error tells the agent to wait or cancel. Then we cancel our demo jobs.

In [4]:
accepted = []
async with connectMcp() as session:
    for n in range(8):
        result = await callTool(session, "create_poster_job", **{**baseArgs, "prompt": f"cap demo {runId} #{n}"})
        if not result["ok"]:
            print(f"request {n + 1} was refused:\n{result['error']}\n")
            break
        accepted.append(result["data"]["job_id"])
    for jobId in accepted:
        await callTool(session, "cancel_job", job_id=jobId)

print(f"{len(accepted)} jobs were accepted before the limit; all cancelled again.")

request 4 was refused:
Error executing tool create_poster_job: 5 jobs are already processing (limit 5). Next step: Wait for running jobs to finish (poll get_job_status) or cancel one, then retry in about 10 seconds.



3 jobs were accepted before the limit; all cancelled again.


## a failed job

To get a failure on demand, this cell starts its own copy of the stack where the first job is forced to fail. Look for three things:

1. the failed result says `retryable: true` and that a retry is safe
2. retrying with the same inputs makes a new job (a failed job doesn't block a retry), and it works
3. the api key never appears in anything the agent sees

In [5]:
from agent.harness import posterStack, demoApiKey

seen = []
async with posterStack(MOCK_FAIL_FIRST_N=1, MOCK_FAIL_RATE=0, MOCK_TIME_SCALE=0.1) as session:
    first = await callTool(session, "create_poster_job", **baseArgs)
    failed = await waitForJob(session, first["data"]["job_id"], pollSeconds=0.5)
    show(failed["data"])

    retry = await callTool(session, "create_poster_job", **baseArgs)
    recovered = await waitForJob(session, retry["data"]["job_id"], pollSeconds=0.5)
    seen = [first, failed, retry, recovered]

print("\nfirst job :", first["data"]["job_id"], "->", failed["data"]["status"])
print("retry job :", retry["data"]["job_id"], "->", recovered["data"]["status"])
assert demoApiKey not in json.dumps(seen), "the API key leaked!"
print("API key found in tool output: no")

poll 1: processing (elapsed 0.0s)


poll 2: processing (elapsed 0.5s)


poll 3: processing (elapsed 1.0s)


poll 4: failed (elapsed 1.5s)
{
  "job_id": "f7d886c15f0e",
  "status": "failed",
  "elapsed_seconds": 1.5,
  "error": {
    "code": "generation_failed",
    "message": "The image model returned an internal error while generating this poster.",
    "retryable": true
  },
  "next_step": "Generation failed. It is safe to call create_poster_job again with the same inputs (a failed job does not block a retry). Retry at most 2 times, then tell the user it failed."
}
poll 1: processing (elapsed 0.0s)


poll 2: processing (elapsed 0.5s)


poll 3: processing (elapsed 1.0s)


poll 4: completed (elapsed 1.5s)



first job : f7d886c15f0e -> failed
retry job : 919c61defc48 -> completed
API key found in tool output: no


Also covered by tests but not shown here: timeouts are retried automatically, a job stuck for over 60 s gets flagged, and a wrong server api key gives a config error without printing the key.

Next: an LLM agent in front of these tools.